# Using Upstage Information Extract on SageMaker Jumpstart

Upstage Information Extract is a powerful API designed to automatically extract structured information from documents. It bundles three capabilities behind a single endpoint:

- **Information Extraction** — Extract structured data from any document, regardless of type or format, using semantic understanding and user-defined schemas.
- **Schema Generation** — Automatically generate an initial JSON schema from up to three sample documents, optionally guided by a system message — saves the trial-and-error of hand-authoring.
- **Document Classification** — Classify a document into one of a user-defined set of categories.

This sample notebook shows you how to deploy Upstage Information Extract using Amazon SageMaker and exercise all three APIs.

## Pre-requisites:

1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
2. Ensure that IAM role used has **AmazonSageMakerFullAccess**
3. To deploy this ML model successfully, ensure that:
   1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used:
      1. **aws-marketplace:ViewSubscriptions**
      2. **aws-marketplace:Unsubscribe**
      3. **aws-marketplace:Subscribe**

## Contents:

1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Setup](#2.-Setup)
3. [Create an endpoint](#3.-Create-an-endpoint)
4. [Perform real-time inference](#4.-Perform-real-time-inference)
   1. [Calling the endpoint](#A.-Calling-the-endpoint)
   2. [API 1: Information Extraction](#B.-API-1:-Information-Extraction)
   3. [API 2: Schema Generation](#C.-API-2:-Schema-Generation)
   4. [API 3: Document Classification](#D.-API-3:-Document-Classification)
5. [Clean-up](#5.-Clean-up)
   1. [Delete the endpoint](#A.-Delete-the-endpoint)
   2. [Delete the model](#B.-Delete-the-model)

## Usage instructions

You can run this notebook one cell at a time (By using Shift+Enter for running a cell).

## Sample files

This notebook expects the following files in the same directory:

- `sample_bank_statement.png` — used for Information Extraction, Schema Generation, and Document Classification.
- `sample_commercial_invoice.jpg` — used for Document Classification.

## 1. Subscribe to the model package

To subscribe to the model package:

<!-- TODO: After public AWS Marketplace release, hyperlink "Upstage Information Extract" below to the Marketplace listing page (e.g., https://aws.amazon.com/marketplace/pp/<product-id>). -->

1. Open the Upstage Information Extract model package listing page on AWS Marketplace.
2. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
3. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms.
4. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using Boto3. Copy the ARN corresponding to your region and specify the same in the following cell.

## 2. Setup

In [ ]:
%pip -q install "sagemaker<3" boto3

In [ ]:
import json
import base64
import boto3
import sagemaker
from sagemaker import ModelPackage

In [ ]:
boto_session = boto3.Session()
sm_session = sagemaker.Session(boto_session=boto_session)

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto_session.client("iam")
    role = [r for r in iam.list_roles()["Roles"] if r["RoleName"].startswith("AmazonSageMaker-ExecutionRole")][0]["Arn"]

In [ ]:
# Select the appropriate model package ARN based on the region.
# TODO: Replace `<account-id>` and add other regions after public AWS Marketplace release.
model_package_name = "information-extract-260304-1"

# Mapping for Model Packages
model_package_map = {
    "us-west-2": f"arn:aws:sagemaker:us-west-2:<account-id>:model-package/{model_package_name}",
}

region = sm_session.boto_region_name
if region not in model_package_map:
    raise Exception(
        f"Current boto3 session region '{region}' is not supported. "
        f"Information Extract is currently only available in: {list(model_package_map.keys())}."
    )

model_package_arn = model_package_map[region]
print(f"Model Package: '{model_package_arn}'")

## 3. Create an endpoint

If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

> **Note:** Initial deployment takes approximately ~10 minutes.

In [ ]:
model_name = "Upstage-Information-Extract"

real_time_inference_instance_type = "ml.g7e.4xlarge"

In [ ]:
# Create a deployable model from the model package.
model = ModelPackage(
    role=role, model_package_arn=model_package_arn, sagemaker_session=sm_session
)

endpoint_name = sagemaker.utils.name_from_base(model_name)
print(f"endpoint name: '{endpoint_name}'")

In [ ]:
# Deploy the model
model.deploy(1, real_time_inference_instance_type, endpoint_name=endpoint_name)

## 4. Perform real-time inference

Information Extract exposes three APIs through a single endpoint. The API to invoke is selected by the `api` field in the request body. All three APIs share:

- `Content-Type: application/json`
- The input document is supplied as a base64-encoded data URL inside `messages[0].content[0].image_url.url`.
- The response uses an OpenAI-style chat-completion shape; the actual result is in `choices[0].message.content`.

Information Extract supports `application/octet-stream` as the MIME type for the data URL, which automatically detects file types (PDF, HWP, JPEG, PNG, etc.) from binary content. This means you do not need to specify the MIME type per file.

### A. Calling the endpoint

The cells below define a file-encoding helper and the SageMaker runtime client used by all three APIs.

In [ ]:
def encode_file_to_data_url(path: str) -> str:
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:application/octet-stream;base64,{b64}"

In [ ]:
runtime_sm_client = boto3.client("runtime.sagemaker")

### B. API 1: Information Extraction

Extract structured fields from a document according to a JSON schema you supply in `response_format`. The response's `choices[0].message.content` is a JSON string that conforms to your schema.

In [ ]:
ie_filepath = "sample_bank_statement.png"
ie_data_url = encode_file_to_data_url(ie_filepath)

ie_payload = {
    "api": "information-extraction",
    "model": "information-extract",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": ie_data_url}}
            ],
        }
    ],
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "document_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "bank_name": {
                        "type": "string",
                        "description": "The name of bank in bank statement",
                    }
                },
            },
        },
    },
}

ie_response = runtime_sm_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(ie_payload),
)
ie_result = json.loads(ie_response["Body"].read())
ie_extracted = json.loads(ie_result["choices"][0]["message"]["content"])
print(json.dumps(ie_extracted, indent=2, ensure_ascii=False))

### C. API 2: Schema Generation

If you do not have a JSON schema ready, the schema-generation API can produce one for you from a document and a short instruction in the system message. The response's `choices[0].message.content` is itself a JSON string that decodes into a `response_format` object — meaning you can plug it directly into the Information Extraction API without any conversion.

In [ ]:
sg_payload = {
    "api": "schema-generation",
    "model": "information-extract",
    "messages": [
        {"role": "system", "content": "Generate schema about bank_name."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": ie_data_url}}
            ],
        },
    ],
}

sg_response = runtime_sm_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(sg_payload),
)
sg_result = json.loads(sg_response["Body"].read())
generated_response_format = json.loads(sg_result["choices"][0]["message"]["content"])
print(json.dumps(generated_response_format, indent=2, ensure_ascii=False))

#### Use the generated schema for extraction

The generated object is already in `response_format` shape, so it can be passed straight back into the Information Extraction API.

In [ ]:
sg_reuse_payload = {
    "api": "information-extraction",
    "model": "information-extract",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": ie_data_url}}
            ],
        }
    ],
    "response_format": generated_response_format,
}

sg_reuse_response = runtime_sm_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(sg_reuse_payload),
)
sg_reuse_result = json.loads(sg_reuse_response["Body"].read())
sg_reuse_extracted = json.loads(sg_reuse_result["choices"][0]["message"]["content"])
print(json.dumps(sg_reuse_extracted, indent=2, ensure_ascii=False))

### D. API 3: Document Classification

Classify a document into one of a user-defined set of categories. The category set is supplied via `response_format` as a `oneOf` of string `const` values, each with a description that helps the model decide.

Unlike the other two APIs, the response's `choices[0].message.content` is a plain string (the chosen class), not a JSON string.

In [ ]:
classification_response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "document-classify",
        "schema": {
            "type": "string",
            "oneOf": [
                {"const": "invoice", "description": "Commercial invoice with itemized charges and billing information"},
                {"const": "receipt", "description": "Receipt showing purchase transaction details"},
                {"const": "contract", "description": "Legal agreement or contract document"},
                {"const": "cv", "description": "Curriculum vitae or resume"},
                {"const": "bank_statement", "description": "Bank account statement showing transaction history and balance"},
                {"const": "others", "description": "Other"},
            ],
        },
    },
}

Classify the bank statement sample (expected label: `bank_statement`).

In [ ]:
filepath = "sample_bank_statement.png"

dc_payload = {
    "api": "document-classification",
    "model": "document-classify",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": encode_file_to_data_url(filepath)}}
            ],
        }
    ],
    "response_format": classification_response_format,
}

dc_response = runtime_sm_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(dc_payload),
)
dc_result = json.loads(dc_response["Body"].read())
label = dc_result["choices"][0]["message"]["content"]
print(f"{filepath} -> label={label}")

Classify the commercial invoice sample (expected label: `invoice`).

In [ ]:
filepath = "sample_commercial_invoice.jpg"

dc_payload = {
    "api": "document-classification",
    "model": "document-classify",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": encode_file_to_data_url(filepath)}}
            ],
        }
    ],
    "response_format": classification_response_format,
}

dc_response = runtime_sm_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(dc_payload),
)
dc_result = json.loads(dc_response["Body"].read())
label = dc_result["choices"][0]["message"]["content"]
print(f"{filepath} -> label={label}")

## 5. Clean-up

### A. Delete the endpoint

Now that you have successfully performed real-time inference, you can delete the endpoint and avoid being charged.

In [ ]:
model.sagemaker_session.delete_endpoint(endpoint_name)
model.sagemaker_session.delete_endpoint_config(endpoint_name)

### B. Delete the model

In [ ]:
model.delete_model()